In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest

# =====================================================================
# STEP 1: LOAD AND CLEAN THE CSV DATA
# =====================================================================
print("Loading data from CSV...")
file_path = r"C:\Users\DELL\Documents\New folder\Call-Center-Escalation-Risk-Prediction-for-Essential-Services\embeddings.csv"
df = pd.read_csv(file_path)

print(f"Dataframe loaded successfully! Shape: {df.shape}")
print("Available columns in your file:", df.columns.tolist())

# =====================================================================
# STEP 2: PARSE THE EMBEDDINGS MATRIX
# =====================================================================
# This safely converts the text-based column into a clean numerical matrix
def parse_embeddings(val):
    if isinstance(val, str):
        try:
            return np.array(ast.literal_eval(val))
        except:
            # Fallback if text formatting has spaces instead of commas
            clean_str = ','.join(val.replace('[', '').replace(']', '').split())
            return np.array(ast.literal_eval(f"[{clean_str}]"))
    return np.array(val)

print("\nExtracting and formatting embedding matrix...")
# NOTE: If your column name is NOT 'embeddings', change 'embeddings' below to match your column
if 'embeddings' in df.columns:
    embeddings = np.vstack(df['embeddings'].apply(parse_embeddings).values)
else:
    embeddings = df.select_dtypes(include=[np.number]).to_numpy()
    print("No 'embeddings' column found. Using numeric dataframe columns as embeddings.")
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(-1, 1)
print(f"Final Embeddings Matrix Shape: {embeddings.shape}")

# =====================================================================
# STEP 3: PART 7 - DBSCAN CLUSTERING
# =====================================================================
print("\nRunning DBSCAN Clustering (Part 7)...")
dbscan = DBSCAN(eps=0.4, min_samples=5, metric='cosine') 
df['cluster_id'] = dbscan.fit_predict(embeddings)

print("Cluster distribution:")
print(df['cluster_id'].value_counts())

# =====================================================================
# STEP 4: PART 8 - ANOMALY DETECTION
# =====================================================================
print("\nRunning Anomaly Detection (Part 8)...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_label'] = iso_forest.fit_predict(embeddings)
df['anomaly_score'] = iso_forest.decision_function(embeddings)

# Map labels for easier reading (1 = anomaly, 0 = normal)
df['is_anomaly'] = df['anomaly_label'].apply(lambda x: 1 if x == -1 else 0)
print(f"Total Anomalies Detected: {df['is_anomaly'].sum()}")

# =====================================================================
# STEP 5: EXPORT PROGRESS FOR PART 9
# =====================================================================
output_path = r"C:\Users\DELL\Documents\New folder\Call-Center-Escalation-Risk-Prediction-for-Essential-Services\dataset_with_clusters_and_anomalies.csv"
df.to_csv(output_path, index=False)
print(f"\nSuccess! File saved at: {output_path}")

Loading data from CSV...
Dataframe loaded successfully! Shape: (1000, 768)
Available columns in your file: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '

KeyError: 'embeddings'